# 04 · Validate

This notebook checks the combined table with rules written in code. A rule either has to pass (`assert`), or the rows it finds are fixed
or kept and recorded with a reason in `docs/corrections.csv`. Two rules compare columns rather than looking at one column at a time: the
country codes from two different sources must agree, and a large jump in the share price index must come with high inflation.

In [1]:
from pathlib import Path

import pandas as pd

INTERIM = Path("../data/interim")
DOCS = Path("../docs")
DOCS.mkdir(exist_ok=True)

In [2]:
combined = pd.read_parquet(INTERIM / "combined.parquet")
checked = combined.copy()

# Start from the rows that 03_combine removed, so docs/corrections.csv has every correction in one place.
corrections = pd.read_csv(INTERIM / "combine_corrections.csv").to_dict("records")
print(len(corrections), "corrections from 03_combine")
checked.shape

47 corrections from 03_combine


(517, 7)

## One row per country and year, in the right range

In [3]:
assert not checked.duplicated(["country_code", "year"]).any()

In [4]:
assert checked["year"].between(2014, 2024).all()
assert (checked["share_price_index"] > 0).all()

## Every country has all 11 years of share prices

In [5]:
n_years = checked.groupby("country_code")["year"].count()
incomplete = n_years[n_years < 11]
incomplete

Series([], Name: year, dtype: int64)

In [6]:
for code in incomplete.index:
    corrections.append({
        "country_code": code,
        "rule": "all 11 years of share prices",
        "action": "country removed",
        "reason": f"only {incomplete[code]} of 11 years in the OECD data",
    })

checked = checked[~checked["country_code"].isin(incomplete.index)]
checked["country_code"].nunique()

47

## Every country has inflation for 2015 to 2024
2014 isn't needed: inflation is applied from 2015 on, because the 2014 index is the starting point.

In [7]:
needed = checked[checked["year"] >= 2015]
no_inflation = needed[needed["inflation_pct"].isna()]
no_inflation

,country_code,alpha2,wb_alpha2,country,year,share_price_index,inflation_pct


In [8]:
for code in no_inflation["country_code"].unique():
    years = no_inflation[no_inflation["country_code"] == code]["year"].tolist()
    corrections.append({
        "country_code": code,
        "rule": "inflation for 2015 to 2024",
        "action": "country removed",
        "reason": f"no World Bank inflation for {years}",
    })

checked = checked[~checked["country_code"].isin(no_inflation["country_code"])]
checked["country_code"].nunique()

47

## The two sources agree on which country each code is
The share prices and the inflation come from different organisations and are joined on the three-letter code. The World Bank also gives
each country its own two-letter code, and the ISO table (from Wikipedia) gives the two-letter code that belongs to each three-letter code.
If the two agree on every row, the join has matched each country with its own inflation.

In [9]:
mismatch = checked[checked["alpha2"] != checked["wb_alpha2"]]
assert mismatch.empty, f"the World Bank and the ISO table disagree on these countries:\n{mismatch}"
print("World Bank and ISO two-letter codes agree for", checked["country_code"].nunique(), "countries")

World Bank and ISO two-letter codes agree for 47 countries


## Inflation is in a plausible range

In [10]:
unusual = checked[(checked["inflation_pct"] > 25) | (checked["inflation_pct"] < -2)]
unusual

,country_code,alpha2,wb_alpha2,country,year,share_price_index,inflation_pct
492,TUR,TR,TR,Türkiye,2022,369.2984,72.308830
493,TUR,TR,TR,Türkiye,2023,785.9030,53.859408
494,TUR,TR,TR,Türkiye,2024,1190.7100,58.506449


In [11]:
for code in unusual["country_code"].unique():
    rows = unusual[unusual["country_code"] == code]
    corrections.append({
        "country_code": code,
        "rule": "inflation between -2% and 25%",
        "action": "kept",
        "reason": f"{len(rows)} year(s) outside the range, highest {rows['inflation_pct'].max():.1f}%. Checked against the World Bank website, the values are real.",
    })

## Large jumps in the share price index come with high inflation
The index follows share prices in local currency, so in a country with high inflation it can rise a lot just because prices in general
rise. This rule compares two columns: a rise of more than 100% in a year is only accepted if inflation that year was above 10%, and a fall
of more than 60% in a year is never accepted without a closer look.

In [12]:
checked = checked.sort_values(["country_code", "year"])
checked["index_change_pct"] = checked.groupby("country_code")["share_price_index"].pct_change() * 100

big_rises = checked[checked["index_change_pct"] > 100]
crashes = checked[checked["index_change_pct"] < -60]

assert crashes.empty, f"falls of more than 60% in a year need a closer look:\n{crashes}"
unexplained = big_rises[big_rises["inflation_pct"] <= 10]
assert unexplained.empty, f"rises of more than 100% without high inflation:\n{unexplained}"
big_rises

,country_code,alpha2,wb_alpha2,country,year,share_price_index,inflation_pct,index_change_pct
493,TUR,TR,TR,Türkiye,2023,785.903,53.859408,112.80975


In [13]:
for _, row in big_rises.iterrows():
    corrections.append({
        "country_code": row["country_code"],
        "rule": "yearly index rise above 100% needs inflation above 10% the same year",
        "action": "kept",
        "reason": (
            f"index rose {row['index_change_pct']:.0f}% in {row['year']} while inflation was {row['inflation_pct']:.1f}%; "
            "the index follows prices in local currency, so a large rise is expected"
        ),
    })

## Save the corrections and the checked table

In [14]:
corrections = pd.DataFrame(corrections, columns=["country_code", "rule", "action", "reason"])
corrections.to_csv(DOCS / "corrections.csv", index=False)
corrections["rule"].value_counts()

rule
World Bank area must be a country in the ISO 3166-1 table               46
OECD area must be a country in the ISO 3166-1 table                      1
inflation between -2% and 25%                                            1
yearly index rise above 100% needs inflation above 10% the same year     1
Name: count, dtype: int64

In [15]:
print(combined["country_code"].nunique(), "countries before,", checked["country_code"].nunique(), "after")

# wb_alpha2 and index_change_pct were only needed for the checks above.
checked = checked.drop(columns=["wb_alpha2", "index_change_pct"]).reset_index(drop=True)
checked.to_parquet(INTERIM / "checked.parquet", index=False)
checked.shape

47 countries before, 47 after


(517, 6)